In [5]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from sklearn.model_selection import train_test_split
from src.preprocessing import clean_data
from src.feature_engineering import create_features

# Load original data
df = pd.read_excel("../data/raw/credit_card_og.xlsx")

# Same split as training
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Clean
train_df = clean_data(train_df)
test_df = clean_data(test_df)

# Remove target
train_df = train_df.drop("Attrition_Flag", axis=1)
test_df = test_df.drop("Attrition_Flag", axis=1)

# Feature engineering
train_df = create_features(train_df)
test_df = create_features(test_df)

# Encode
X_train = pd.get_dummies(train_df, drop_first=True)
X_test = pd.get_dummies(test_df, drop_first=True)

# Align columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [2]:
pip install shap

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import os
import joblib

# Ensure save path exists
os.makedirs("../dashboard/static", exist_ok=True)

model = joblib.load("../models/xgboost.pkl")

# ----------------------------
# Prepare data (important)
# ----------------------------
X_sample = X_test.sample(n=300, random_state=42)

# Remove any unwanted columns (safety)
X_sample = X_sample[[col for col in X_sample.columns if "Quarter_none" not in col]]

# ----------------------------
# SHAP Explainer
# ----------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# ----------------------------
# SHAP Summary Plot (Top features only)
# ----------------------------
shap.summary_plot(
    shap_values,
    X_sample,
    max_display=10,   # only top 10 features
    show=False
)

plt.title("Top Drivers of Customer Risk")

plt.savefig("../dashboard/static/shap.png")
plt.close()